# Etapa de Ingeniería de atributos

In [ ]:
```python
"""
Script de Ingeniería de Atributos para predicción de casos de dengue
(Sin creación de rezagos, solo transformaciones y limpieza)
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import os

# ============================================================
# CONFIGURACIÓN
# ============================================================
RUTA_XLSX = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\2_datos\1_raw\2_meteo_epi_2021-2026_1_rezagos.xlsx"
RUTA_SALIDA = r"C:\Users\marco\Documentos\investigacion\machine_learning_idalina\6_redes_neuronales\1_red_neuronal_1_sklearn_mlp\1_scripts\modelo_red_neuronal_MLP_y_secuencia_aprendizaje\1_dataset_con_ingenieria_atributos"

# Crear directorio si no existe
os.makedirs(RUTA_SALIDA, exist_ok=True)

VARIABLES_BASE = [
    "temp", "temp_max", "temp_min", "hum_esp", "hum_rel",
    "prec", "dias_lluvia", "vel_vi", "vel_vi_max", "vel_vi_min",
    "soi", "sst",
]

N_LAGS = 12
UMBRAL_SOI = 50
PROPORCION_TRAIN = 0.90  # 90% entrenamiento, 10% prueba

# ============================================================
# 1. CARGA Y EXPLORACIÓN INICIAL
# ============================================================
print("=" * 60)
print("INGENIERÍA DE ATRIBUTOS - DATOS DE DENGUE")
print("=" * 60)

df = pd.read_excel(RUTA_XLSX)
df["fecha"] = pd.to_datetime(df["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)

print(f"\nDatos cargados: {len(df)} registros")
print(f"Rango de fechas: {df['fecha'].min().date()} a {df['fecha'].max().date()}")
print(f"Variables disponibles: {df.shape[1]}")

# ============================================================
# 2. LIMPIEZA DE OUTLIERS EN SOI
# ============================================================
print("\n" + "=" * 60)
print("LIMPIEZA DE OUTLIERS EN VARIABLE SOI")
print("=" * 60)

soi_cols = ["soi"] + [f"soi_lag_{i}" for i in range(1, N_LAGS + 1)]

# Contar outliers antes de la limpieza
n_afectados = (df[soi_cols].abs() > UMBRAL_SOI).sum().sum()
print(f"Celdas fuera de rango detectadas: {n_afectados}")

# Limpiar valores extremos
for c in soi_cols:
    df.loc[df[c].abs() > UMBRAL_SOI, c] = np.nan

# Interpolar valores nulos
df[soi_cols] = df[soi_cols].interpolate(method="linear", limit_direction="both")

print("✓ Valores fuera de rango reemplazados por interpolación lineal")
print("  (SOI típico: entre -40 y +35)")

# ============================================================
# 3. INGENIERÍA DE ATRIBUTOS - VARIABLES DERIVADAS
# ============================================================
print("\n" + "=" * 60)
print("CREACIÓN DE VARIABLES DERIVADAS")
print("=" * 60)

df_fe = df.copy()

# 3.1 Variables estacionales (componentes cíclicos)
df_fe["mes"] = df_fe["fecha"].dt.month
df_fe["semana"] = df_fe["fecha"].dt.isocalendar().week
df_fe["dia_del_ano"] = df_fe["fecha"].dt.dayofyear

# Convertir variables cíclicas a seno/coseno para mantener la estacionalidad
df_fe["seno_mes"] = np.sin(2 * np.pi * df_fe["mes"] / 12)
df_fe["coseno_mes"] = np.cos(2 * np.pi * df_fe["mes"] / 12)
df_fe["seno_semana"] = np.sin(2 * np.pi * df_fe["semana"] / 52)
df_fe["coseno_semana"] = np.cos(2 * np.pi * df_fe["semana"] / 52)

# 3.2 Variables climáticas derivadas
# Amplitud térmica diaria
df_fe["amplitud_termica"] = df_fe["temp_max"] - df_fe["temp_min"]

# Índice combinado de humedad y temperatura (sensación térmica aproximada)
df_fe["hum_temp_index"] = df_fe["hum_rel"] * df_fe["temp"] / 100

# Precipitación acumulada en ventanas móviles (últimas 4 y 8 semanas)
df_fe["prec_4s_acum"] = df_fe["prec"].rolling(window=4, min_periods=1).sum()
df_fe["prec_8s_acum"] = df_fe["prec"].rolling(window=8, min_periods=1).sum()

# Días de lluvia acumulados en ventanas móviles
df_fe["dias_lluvia_4s_acum"] = df_fe["dias_lluvia"].rolling(window=4, min_periods=1).sum()
df_fe["dias_lluvia_8s_acum"] = df_fe["dias_lluvia"].rolling(window=8, min_periods=1).sum()

# 3.3 Variables de tendencia climática
# Diferencia de temperatura con respecto a la semana anterior
df_fe["temp_diff_lag1"] = df_fe["temp"].diff()

# Diferencia de precipitación con respecto a la semana anterior
df_fe["prec_diff_lag1"] = df_fe["prec"].diff()

# 3.4 Variable de temporada de dengue (febrero-mayo es pico en muchos países)
df_fe["temporada_dengue"] = df_fe["mes"].isin([2, 3, 4, 5]).astype(int)

# 3.5 Variables de casos de dengue (transformaciones)
# Log transformación para estabilizar varianza (evita que valores muy altos dominen)
df_fe["log_casos_dengue"] = np.log1p(df_fe["casos_dengue"])

# 3.6 Tasa de cambio de casos (crecimiento semanal)
df_fe["casos_dengue_diff"] = df_fe["casos_dengue"].diff()
df_fe["casos_dengue_pct_change"] = df_fe["casos_dengue"].pct_change() * 100

# 3.7 Promedio móvil de casos (suavizado)
df_fe["casos_dengue_ma4"] = df_fe["casos_dengue"].rolling(window=4, min_periods=1).mean()
df_fe["casos_dengue_ma8"] = df_fe["casos_dengue"].rolling(window=8, min_periods=1).mean()

print("Variables creadas:")
print(f"  - Estacionales: mes, semana, seno_mes, coseno_mes, seno_semana, coseno_semana")
print(f"  - Climáticas derivadas: amplitud_termica, hum_temp_index")
print(f"  - Acumulados: prec_4s_acum, prec_8s_acum, dias_lluvia_4s_acum, dias_lluvia_8s_acum")
print(f"  - Tendencias: temp_diff_lag1, prec_diff_lag1")
print(f"  - Temporada: temporada_dengue")
print(f"  - Transformaciones casos: log_casos_dengue, casos_dengue_diff, casos_dengue_pct_change")
print(f"  - Promedios móviles: casos_dengue_ma4, casos_dengue_ma8")
print(f"\nTotal de variables: {df_fe.shape[1]}")

# ============================================================
# 4. DIVISIÓN TEMPORAL (90% entrenamiento, 10% prueba)
# ============================================================
print("\n" + "=" * 60)
print("DIVISIÓN TEMPORAL DE DATOS")
print("=" * 60)

n = len(df_fe)
corte = int(n * PROPORCION_TRAIN)

train_df = df_fe.iloc[:corte].copy()
test_df = df_fe.iloc[corte:].copy()

print(f"Entrenamiento: {train_df['fecha'].min().date()} a {train_df['fecha'].max().date()}")
print(f"  {len(train_df)} semanas ({len(train_df)/n*100:.1f}%)")
print(f"Prueba: {test_df['fecha'].min().date()} a {test_df['fecha'].max().date()}")
print(f"  {len(test_df)} semanas ({len(test_df)/n*100:.1f}%)")

# ============================================================
# 5. GUARDADO DE DATASETS
# ============================================================
print("\n" + "=" * 60)
print("GUARDADO DE DATASETS")
print("=" * 60)

# Guardar datasets completos
archivo_train = os.path.join(RUTA_SALIDA, "train_dataset_fe.xlsx")
archivo_test = os.path.join(RUTA_SALIDA, "test_dataset_fe.xlsx")

train_df.to_excel(archivo_train, index=False)
test_df.to_excel(archivo_test, index=False)

print(f"✓ Dataset de entrenamiento guardado en:")
print(f"  {archivo_train}")
print(f"✓ Dataset de prueba guardado en:")
print(f"  {archivo_test}")

# ============================================================
# 6. RESUMEN FINAL
# ============================================================
print("\n" + "=" * 60)
print("RESUMEN DE VARIABLES FINALES")
print("=" * 60)

# Listar todas las variables disponibles
print("\nVariables disponibles para el modelo:")
for i, col in enumerate(train_df.columns, 1):
    print(f"  {i:3d}. {col}")

print(f"\nTotal: {len(train_df.columns)} variables")
print(f"Filas de entrenamiento: {len(train_df)}")
print(f"Filas de prueba: {len(test_df)}")

print("\n" + "=" * 60)
print("¡INGENIERÍA DE ATRIBUTOS COMPLETADA CON ÉXITO!")
print("=" * 60)
```